# Chapter 10: Generate, check and audit a grounded subsection

Chapter 10 treats report drafting as a test with a declared instruction space, discovery packets, a frozen selection and a reserved packet. In this run a language model drafted nine report subsections from source-checked packets. This notebook reads the automatic coverage results and then the separate review of the drafted text. The live reporting experiment varies coverage and qualification instructions. It freezes a choice on two discovery packets before applying it to a reserved packet. Identifier coverage is measured automatically; read the actual text before treating it as faithful reporting.

**Evidence.** Run 20260923T220909Z repeated the regional study end to end on two machines. `spark-fe12` ran numerical discovery with three new optimizer seeds (74101 to 74103), one live structural proposal and a fresh 200,000-observation confirmation sample. `spark-ef84` ran 70 live answer tests and nine report drafts with `claude-sonnet-5`. The population and frozen logistic predictor are those of the chapter companions; the seeds, proposal and confirmation sample are new, so these numbers should not be combined with the historical run's.

## Setup and source checks

The setup cell locates the data root, the directory whose relative layout the evidence records use. It uses `MODEL_VALIDATION_ROOT` when that variable is set. Otherwise it searches for a repository checkout containing the current directory and then for the copy installed with the `aiv` package. `R` is this run's folder under the data root, and `read` loads a JSON file from it.

`reference.json` was exported from the run's regional results and records their SHA-256 digest. The final assertion recomputes that digest, so every value used below comes from the unchanged results of this run. This notebook reads recorded outputs; rerunning it makes no language-model call.

In [1]:
from pathlib import Path
import json, hashlib, os, platform
import numpy as np
import pandas as pd

# Locate the data root: the directory whose relative layout the evidence records use.
MARKER = "book/notebooks/regional_case/companion.py"

def find_data_root():
    configured = os.environ.get("MODEL_VALIDATION_ROOT")
    if configured:
        root = Path(configured).expanduser().resolve()
        if not (root / MARKER).is_file():
            raise FileNotFoundError(f"MODEL_VALIDATION_ROOT must contain {MARKER}")
        return root, "MODEL_VALIDATION_ROOT"
    for path in (Path.cwd(), *Path.cwd().parents):
        if (path / MARKER).is_file():
            return path, "repository checkout"
    try:
        from aiv.companions import locate
    except ImportError:
        raise FileNotFoundError(
            "Companion data not found. Install the aiv package, open the notebook "
            "inside the repository checkout, or set MODEL_VALIDATION_ROOT.") from None
    return locate()

ROOT, DATA_SOURCE = find_data_root()
LIVE = ROOT / "book/experiments/regional_live"
R = ROOT / "book/experiments/regional_live/runs/20260923T220909Z"
print("Python:", platform.python_version())
print("Data root found from:", DATA_SOURCE)
print("Run:", R.relative_to(ROOT).as_posix())
read=lambda p: json.loads((R/p).read_text(encoding="utf-8"))
results=read('regional/results.json')
reference=read('reference.json')
seeds=reference['seeds']
assert hashlib.sha256((R/'regional/results.json').read_bytes()).hexdigest()==reference['source_sha256']


Python: 3.12.3
Data root found from: repository checkout
Run: book/experiments/regional_live/runs/20260923T220909Z


## Automatic coverage and selection

Four instructions crossed two binary factors: whether to keep only the most useful measurements or include all supplied ones, and whether to keep only necessary qualifications or preserve every scope qualification. Each instruction drafted a subsection for the two discovery packets. A draft is `valid` when it cites every required fact and qualification identifier, and its `cost` counts the identifiers it included. The selection rule froze the instruction that was valid on both discovery packets with the smallest total cost, breaking ties by ID. The frozen instruction was then applied once to the reserved packet for seed 74103. The cell displays the selected instruction, the nine drafts' coverage results and the reserved draft's text.

In [2]:
reports=read('report_results.json')
display(reports['selected'])
display(pd.DataFrame([{k:v for k,v in row.items() if k!='plan'} for row in reports['rows']]))
for row in reports['rows']:
 if row['phase']=='reserved':
  print(row.get('plan',{}).get('text',row.get('error')))
print('Automatic validity means identifier coverage only, not semantic faithfulness.')

{'id': 'w1',
 'text': 'Keep only the most useful measurements. Preserve every scope qualification. '}

,instruction,packet,phase,valid,cost,scope
0,w0,seed_74101,discovery,False,10,Identifier coverage only; text faithfulness re...
1,w0,seed_74102,discovery,True,11,Identifier coverage only; text faithfulness re...
2,w1,seed_74101,discovery,True,11,Identifier coverage only; text faithfulness re...
3,w1,seed_74102,discovery,True,11,Identifier coverage only; text faithfulness re...
4,w2,seed_74101,discovery,False,9,Identifier coverage only; text faithfulness re...
5,w2,seed_74102,discovery,False,9,Identifier coverage only; text faithfulness re...
6,w3,seed_74101,discovery,True,11,Identifier coverage only; text faithfulness re...
7,w3,seed_74102,discovery,True,11,Identifier coverage only; text faithfulness re...
8,w1,seed_74103,reserved,True,11,Identifier coverage only; text faithfulness re...


This subsection summarizes the reserved-phase confirmation metrics for seed 74103 across its two evaluated regions, reflection and continuation. For the reflection region, the Brier score was 0.0844 [region:confirmation/reflection_74103/brier], with a contrast (squared probability error) of 0.0709 [region:confirmation/reflection_74103/contrast] and a sample share of 10.59% [region:confirmation/reflection_74103/share]. For the continuation region, the Brier score was 0.0854 [region:confirmation/continuation_74103/brier], with a contrast of 0.0718 [region:confirmation/continuation_74103/contrast] and a sample share of 10.28% [region:confirmation/continuation_74103/share]. The two regions show closely comparable error and contrast magnitudes, suggesting no meaningful divergence in confirmation-stage predictive behavior between reflection and continuation subsets at this seed.

Several scope qualifications constrain interpretation of these figures. The base predictor underlying these metri

**Read the output.** Instruction `w1`, which keeps the most useful measurements and preserves every qualification, was selected. Instructions `w1` and `w3` were valid on both discovery packets at equal cost, and the tie went to the earlier ID. Six of the nine drafts passed identifier coverage, including the reserved draft. The reserved text cites every identifier and repeats the qualifications, but it also states that the two regions show "no meaningful divergence" in predictive behavior. The packet supplies point estimates with no materiality criterion or uncertainty, so that sentence is unsupported. Identifier coverage cannot detect this kind of error.

## Review the text separately

After the automatic selection was frozen, each of the nine drafts was read against its packet. The review records a verdict and specific findings for every draft. The raw drafts are unchanged, and the review is a post-run qualitative audit, not a preregistered semantic benchmark.

In [3]:
audit=read('report_semantic_review.json')
display(pd.DataFrame(audit['rows']))
print(audit['scope'])
print(audit['interpretation'])

,call,verdict,findings
0,discovery_w0_seed_74101,needs_revision,[Brier score alone does not establish comparab...
1,discovery_w0_seed_74102,needs_revision,[Calls positive error contrast a predictive ed...
2,discovery_w1_seed_74101,no_substantive_error_identified,[Values are correctly rounded and qualificatio...
3,discovery_w1_seed_74102,needs_revision,[Defines contrast as control-vs-treatment erro...
4,discovery_w2_seed_74101,needs_revision,[Calls lower contrast better. This search maxi...
5,discovery_w2_seed_74102,needs_revision,[Infers comparable predictive calibration from...
6,discovery_w3_seed_74101,needs_revision,[Converts absence of variance estimates in the...
7,discovery_w3_seed_74102,no_substantive_error_identified,[Exact supplied numbers and comparison directi...
8,reserved_w1_seed_74103,needs_revision,[Claims no meaningful divergence from close po...


Post-run qualitative audit, not a preregistered or independent semantic benchmark. Raw model drafts are unchanged. Identifier-based selection was frozen before this review.
Seven of nine drafts need substantive revision. Two had no substantive error identified in this review; this is not a certification. Six of nine passed identifier coverage, including the reserved draft that made an unsupported materiality statement.


**Read the output.** Seven of the nine drafts need substantive revision, including the reserved draft. The recorded findings include treating a lower contrast as better although the search maximized contrast, inferring comparable calibration from Brier scores alone and claiming no meaningful divergence without a criterion. Two drafts had no substantive error identified, which is not a certification of those drafts. Automatic coverage and text faithfulness therefore measured different properties here, and the second required reading the drafts.